# Newton root-finder for ppm_Boron using OpenMC tally derivatives (one run per Newton iter).

In [ ]:
#!/usr/bin/env python3
"""
search_with_derivs.py 
"""

import os
import math
import h5py
import openmc
import numpy as np

# Constants
N_A = 6.02214076e23     # Avogadro's number (atoms/mol)
A_B_nat = 10.81         # g/mol approximate atomic mass for natural boron

# ===============================================================
# Helper: automatically find all cell IDs filled with a material
# ===============================================================
def find_cells_using_material(geometry, material):
    """Return list of cell IDs using the given material object."""
    return [
        c.id
        for c in geometry.get_all_cells().values()
        if c.fill is material
    ]
    
# -------------------------
# Model builder (your model, adjusted)
# -------------------------
def build_model(ppm_Boron):
    # Create the pin materials
    fuel = openmc.Material(name='1.6% Fuel', material_id=1)
    fuel.set_density('g/cm3', 10.31341)
    fuel.add_element('U', 1., enrichment=1.6)
    fuel.add_element('O', 2.)

    zircaloy = openmc.Material(name='Zircaloy', material_id=2)
    zircaloy.set_density('g/cm3', 6.55)
    zircaloy.add_element('Zr', 1.)

    water = openmc.Material(name='Borated Water', material_id=3)
    water.set_density('g/cm3', 0.741)
    water.add_element('H', 2.)
    water.add_element('O', 1.)

    # Include amount of boron in the water based on ppm by mass (neglecting other constituents)
    # add_element takes a fraction; here we pass ppm * 1e-6 (mass fraction)
    water.add_element('B', ppm_Boron * 1e-6)

    # Instantiate a Materials object
    materials = openmc.Materials([fuel, zircaloy, water])

    # Create cylinders for the fuel and clad
    fuel_outer_radius = openmc.ZCylinder(r=0.39218)
    clad_outer_radius = openmc.ZCylinder(r=0.45720)

    # Create boundary planes to surround the geometry
    min_x = openmc.XPlane(x0=-0.63, boundary_type='reflective')
    max_x = openmc.XPlane(x0=+0.63, boundary_type='reflective')
    min_y = openmc.YPlane(y0=-0.63, boundary_type='reflective')
    max_y = openmc.YPlane(y0=+0.63, boundary_type='reflective')

    # Create fuel Cell
    fuel_cell = openmc.Cell(name='1.6% Fuel')
    fuel_cell.fill = fuel
    fuel_cell.region = -fuel_outer_radius

    # Create a clad Cell
    clad_cell = openmc.Cell(name='1.6% Clad')
    clad_cell.fill = zircaloy
    clad_cell.region = +fuel_outer_radius & -clad_outer_radius

    # Create a moderator Cell
    moderator_cell = openmc.Cell(name='1.6% Moderator')
    moderator_cell.fill = water
    moderator_cell.region = +clad_outer_radius & (+min_x & -max_x & +min_y & -max_y)

    # Create root Universe
    root_universe = openmc.Universe(name='root universe', universe_id=0)
    root_universe.add_cells([fuel_cell, clad_cell, moderator_cell])

    # Create Geometry and set root universe
    geometry = openmc.Geometry(root_universe)

    # Finish with the settings file
    settings = openmc.Settings()
    settings.batches = 20
    settings.inactive = 10
    settings.particles = 1000
    settings.run_mode = 'eigenvalue'
    settings.verbosity = 1

    # Create an initial uniform spatial source distribution over fissionable zones
    bounds = [-0.63, -0.63, -10, 0.63, 0.63, 10.]
    uniform_dist = openmc.stats.Box(bounds[:3], bounds[3:], only_fissionable=True)
    settings.source = openmc.source.Source(space=uniform_dist)

    model = openmc.model.Model(geometry, materials, settings)
    return model

# -------------------------
# Helper: find tally HDF5 group by name (robust)
# -------------------------
def _find_tally_group_by_name(h5_tallies_group, wanted_name):
    for key in h5_tallies_group:
        if not key.startswith('tally '):
            continue
        g = h5_tallies_group[key]
        # try attribute 'name' first
        name = None
        if 'name' in g.attrs:
            try:
                name = g.attrs['name'].decode()
            except Exception:
                name = g.attrs['name']
        # fallback dataset 'name' inside group
        if name is None:
            if 'name' in g:
                try:
                    name = g['name'][()].decode()
                except Exception:
                    name = g['name'][()]
        if name == wanted_name:
            print("FOUND NAME IN TALLY: ", name)
            return g
    raise RuntimeError(f'Tally named "{wanted_name}" not found in statepoint file')

# -------------------------
# Safe dataset helpers (robust decoding)
# -------------------------
def _to_str(x):
    """Convert HDF5 bytes/arrays to python str in a forgiving way."""
    try:
        if isinstance(x, bytes):
            return x.decode()
        import numpy as _np
        if hasattr(x, 'dtype') and x.dtype == _np.bytes_:
            return x.astype(str).item()
    except Exception:
        pass
    try:
        return str(x)
    except Exception:
        return None

def _read_dataset_safe(group, name):
    """Return dataset value or None, decoding bytes if needed."""
    if name not in group:
        return None
    val = group[name][()]
    if isinstance(val, (bytes, bytearray)):
        try:
            return val.decode()
        except Exception:
            return val
    try:
        import numpy as _np
        if hasattr(val, 'shape') and val.shape == ():
            return val.item()
        if hasattr(val, 'dtype') and val.dtype == _np.bytes_:
            return val.astype(str).item()
    except Exception:
        pass
    return val

# -------------------------
# Helper: extract scalar mean value from a tally group or derivative subgroup
# -------------------------
def _extract_scalar_mean_from_group(group):
    if 'mean' in group:
        arr = group['mean'][()]
        return float(arr) if getattr(arr, 'size', 1) == 1 else float(arr.flatten()[0])
    if 'results' in group:
        arr = group['results'][()]
        return float(arr) if getattr(arr, 'size', 1) == 1 else float(arr.flatten()[0])
    if 'sum' in group:
        arr = group['sum'][()]
        return float(arr) if getattr(arr, 'size', 1) == 1 else float(arr.flatten()[0])
    raise RuntimeError('No recognizable mean/results dataset in HDF5 group')

# -------------------------
# Run OpenMC and pull k, F, A, dF_dN_total, dA_dN_total, rho_water
# -------------------------
def run_once_and_get_derivs(ppm_B, target_batches=300, water_material_id=3,
                            boron_nuclides=('B10', 'B11')):
    """
    Build model, add tallies + derivative requests, run OpenMC,
    and return:
        (k_mean, F_base, A_base, dF_dN_total, dA_dN_total, rho_water)
    """
    # Build model
    model = build_model(ppm_B)
    model.settings.track_generation = True
    

    # Print cells for debug
    print("CELLS IN MODEL:")
    for c in model.geometry.get_all_cells().values():
        print(c.id, c.name)

    # Auto-detect moderator cells
    material_dict = {m.id: m for m in model.materials}
    if water_material_id not in material_dict:
        raise RuntimeError(
            f"Material with ID {water_material_id} not found in model.materials"
        )
    water = material_dict[water_material_id]
    moderator_cell_ids = find_cells_using_material(model.geometry, water)
    if not moderator_cell_ids:
        raise RuntimeError(
            f"Could not locate moderator cells containing material ID {water_material_id}"
        )
    print("AUTO-DETECTED moderator cell IDs:", moderator_cell_ids)
    moderator_filter = openmc.CellFilter(moderator_cell_ids)

    # Base tallies
    tF_base = openmc.Tally(name='FissionBase')
    tF_base.scores = ['nu-fission']
    tA_base = openmc.Tally(name='AbsorptionBase')
    tA_base.scores = ['absorption']

    # Derivative tallies (one pair per nuclide)
    deriv_tallies = []
    for nuc in boron_nuclides:
        deriv = openmc.TallyDerivative(variable='nuclide_density',
                                       material=water_material_id,
                                       nuclide=nuc)

        tf = openmc.Tally(name=f'Fission_deriv_{nuc}')
        tf.scores = ['nu-fission']
        tf.derivative = deriv
        tf.filters = [moderator_filter]

        ta = openmc.Tally(name=f'Absorp_deriv_{nuc}')
        ta.scores = ['absorption']
        ta.derivative = deriv
        ta.filters = [moderator_filter]

        deriv_tallies += [tf, ta]

    model.tallies = openmc.Tallies([tF_base, tA_base] + deriv_tallies)

    # Run OpenMC
    model.settings.batches = target_batches
    model.settings.inactive = max(1, int(target_batches * 0.0667))

    model.run()
    sp_filename = f"statepoint.{target_batches}.h5"

    if not os.path.exists(sp_filename):
        raise RuntimeError("Statepoint file missing after run")

    # Load keff via high-level API
    sp = openmc.StatePoint(sp_filename)
    k_mean = sp.k_combined.nominal_value

    # Open HDF5 for low-level inspection
    with h5py.File(sp_filename, 'r') as f:
        tallies_grp = f['tallies']

        # Base totals
        gF = _find_tally_group_by_name(tallies_grp, 'FissionBase')
        gA = _find_tally_group_by_name(tallies_grp, 'AbsorptionBase')
        F_base = _extract_scalar_mean_from_group(gF)
        A_base = _extract_scalar_mean_from_group(gA)

        # Extract rho_water from materials group (in g/cm3)
        mats_grp = f.get("materials", None)
        rho_water = None
        if mats_grp is not None:
            mk = f"material {water_material_id}"
            if mk in mats_grp:
                mg = mats_grp[mk]
                if "density" in mg:
                    try:
                        rho_water = float(mg["density"][()])
                    except Exception:
                        try:
                            s = _to_str(mg['density'][()])
                            rho_water = float(''.join(ch for ch in s if (ch.isdigit() or ch in ".-")))
                        except Exception:
                            rho_water = None

        if rho_water is None:
            rho_water = build_model(ppm_B).materials[2].density

    # NEW APPROACH: Use OpenMC's Python API to read derivative tallies
    print("Reading derivative tallies using OpenMC Python API...")
    
    dF_dN_total = 0.0
    dA_dN_total = 0.0
    
    for nuc in boron_nuclides:
        fission_tally_name = f'Fission_deriv_{nuc}'
        absorp_tally_name = f'Absorp_deriv_{nuc}'
        
        try:
            # Get fission derivative tally
            fission_tally = sp.get_tally(name=fission_tally_name)
            if fission_tally is not None:
                # Sum over all filter bins to get total derivative
                fission_deriv = float(np.sum(fission_tally.mean))
                dF_dN_total += fission_deriv
                print(f"Found {fission_tally_name}: {fission_deriv:.6e}")
            else:
                print(f"WARNING: Tally {fission_tally_name} not found")
                
            # Get absorption derivative tally  
            absorp_tally = sp.get_tally(name=absorp_tally_name)
            if absorp_tally is not None:
                # Sum over all filter bins to get total derivative
                absorp_deriv = float(np.sum(absorp_tally.mean))
                dA_dN_total += absorp_deriv
                print(f"Found {absorp_tally_name}: {absorp_deriv:.6e}")
            else:
                print(f"WARNING: Tally {absorp_tally_name} not found")
                
        except Exception as e:
            print(f"Error reading derivative tally for {nuc}: {e}")

    print(f"Final derivatives: dF_dN_total = {dF_dN_total:.6e}, dA_dN_total = {dA_dN_total:.6e}")

    return k_mean, F_base, A_base, dF_dN_total, dA_dN_total, rho_water

# -------------------------
# Newton loop: ppm parameter search
# -------------------------
def newton_search_for_ppm(ppm0, k_target, tol=1e-4, max_iter=8,
                          water_material_id=3, boron_nuclides=('B10', 'B11'),
                          damping=0.8, target_batches=300):
    ppm = float(ppm0)
    for it in range(max_iter):
        print(f'\n=== Newton iter {it} | ppm = {ppm:.6g} ===')
        k, F, A, dF_dN, dA_dN, rho_water = run_once_and_get_derivs(
            ppm, target_batches=target_batches,
            water_material_id=water_material_id,
            boron_nuclides=boron_nuclides)
        print(f'  k = {k:.6g}, F = {F:.6g}, A = {A:.6g}')
        print(f'  dF/dN_total = {dF_dN:.6e}, dA/dN_total = {dA_dN:.6e}, rho_water = {rho_water:.6g} g/cm3')

        err = k - k_target
        if abs(err) < tol:
            print(f'Converged: ppm = {ppm:.6g}, k = {k:.6g}')
            return ppm

        # dk/dN = (dF - k * dA) / A
        dk_dN = (dF_dN - k * dA_dN) / A

        # convert dN/dppm (ppm by mass -> number density in atoms/cm3 per ppm)
        dN_dppm = 1e-6 * rho_water * N_A / A_B_nat
        dk_dppm = dk_dN * dN_dppm

        print(f'  dk/dN = {dk_dN:.6e}, dN/dppm = {dN_dppm:.6e}, dk/dppm = {dk_dppm:.6e}')

        if abs(dk_dppm) < 1e-20:
            raise RuntimeError('Derivative too small (|dk/dppm| ~ 0). Consider finite-difference fallback.')

        # Newton step (damped)
        delta = err / dk_dppm
        ppm_new = ppm - damping * delta

        # Safety limits
        ppm_new = max(0.0, ppm_new)
        ppm_new = min(ppm_new, 1e6)

        print(f'  delta_ppm (raw) = {delta:.6e}, ppm_new (damped+clamped) = {ppm_new:.6g}')
        ppm = ppm_new

    raise RuntimeError('Newton did not converge within max iterations')

# -------------------------
# Example run (adjust start and target)
# -------------------------
if __name__ == '__main__':
    # initial guess and target k
    ppm_start = 1000.0
    k_target = 1.0

    # You can adjust batches/particles in build_model() or override target_batches here:
    try:
        ppm_solution = newton_search_for_ppm(ppm_start, k_target,
                                             tol=1e-4, max_iter=6,
                                             damping=0.6, target_batches=300)
        print(f'\nFound ppm = {ppm_solution:.6g}')
    except Exception as e:
        print('Search failed:', e)